# China → Mexico Machinery Imports — Data Repair and Exploratory Analysis

*Turning a two-system export into a defensible analysis file: shipments from three Chinese
regions to a Mexican inland destination, 2023–2025.*

> **! Provenance.** Every company, supplier, shipment, price and document in this repository is
> **synthetic**. The four CSVs in `data/raw/` were generated to model a real import operation —
> defects included, deliberately. No real company, supplier or transaction is represented, and
> any resemblance to an existing firm is coincidental.

## Context

A fictional Guadalajara-based importer/distributor buys industrial machinery from suppliers
across three Chinese regions — Guangdong, Zhejiang and Jiangsu — ships it through Manzanillo,
Lázaro Cárdenas or Veracruz, and pays duty, IVA, customs brokerage, drayage and — when clearance
goes wrong — demurrage before the goods reach the warehouse.

The operations file is not one system's output. It is the **merge of the ERP export and the
customs broker's portal**, and the two systems disagree about keys, dates, currency formatting
and port names. Nobody has reconciled them.

## The data

| File | Grain | Lines | Note |
| --- | --- | --- | --- |
| `shipments.csv` | one line per shipment | 1,458 | Latin-1 encoded. Orders 01/2023–12/2025 |
| `products.csv` | one line per SKU | 45 | UTF-8 with BOM. HS-code master |
| `suppliers.csv` | one line per supplier | 15 | Factory city, load port, coordinates |
| `fx_rates.csv` | one line per month | 44 | Joins on the clearance month |

## Scope

**This notebook: data repair and exploratory analysis.** Load → diagnose → repair → document →
explore.

**Out of scope:** any predictive model. Modelling is notebook 2.

## Done when

1. Every date column parses to one unambiguous type, and the ambiguous rows are resolved by
   evidence, not assumption.
2. Every key — `supplier_id`, `sku`, `hs_code` — resolves to a master row, or is recorded as
   unresolvable with a reason.
3. Every money and weight column is numeric, one currency per column, and the invariants hold
   (`fob_total_usd = quantity × unit_price_usd`, `gross_weight_kg ≥ net_weight_kg`).
4. Each duplicate class is identified and a single decision is recorded for it.
5. A **findings table** lists every defect class, the lines it affected, and the decision taken.
6. | *Plain-English comments* | Explain what a line or group of lines does, for non-technical readers |

## Reading this notebook

Two kinds of comments appear in the code cells.

**Plain-English comments** explain what a line or a section of code does, for readers who do
not write code. They carry no marker.

**Tagged annotations** follow the
[Better Comments](https://marketplace.visualstudio.com/items?itemName=aaron-bond.better-comments)
convention. They are reserved for findings, decisions and takeaways — never for mechanics:

| Marker | Meaning |
| --- | --- |
| `# !` | A trap, defect or risk found in the data |
| `# ?` | An open question, or a decision that needs a human call |
| `# *` | The takeaway — what this taught me |
| `# TODO` | Work still outstanding |
| `# //` | An approach tried and rejected, kept for the record |

## SETUP

In [1]:
#
# In plain terms: this cell opens the toolbox, decides where the files live, and
# checks that all four source files are actually there before we analyse anything.
# If a file is missing it stops here with a clear message, instead of letting the
# cells below produce empty results that look like real findings.
# =============================================================================

# --- Python built-ins (nothing to install) -----------------------------------
# sys = python version, os = file paths, io = in-memory text, re = pattern matching,
# json = read JSON files, math = distance calculations, time = stopwatches,
# random = repeatable random numbers, warnings = quiet library notices,
# unicodedata = strip accents from names like "Lázaro Cárdenas".
# One `import` statement can bring in several modules at once, separated by commas.
import sys, os, io, re, json, math, time, random, warnings, unicodedata

# `as dt` renames the module inside this notebook, so we can write dt.date instead of
# datetime.date. Nothing changes except the typing.
import datetime as dt                 # dates: order, shipment, arrival, clearance

# `from X import Y` takes one single thing out of a module instead of the whole module.
# Here it is Path, which builds file paths that work on Windows, macOS and Linux alike.
from pathlib import Path              # file paths that work on any operating system

# --- Data handling ------------------------------------------------------------
# `as np` is just a short nickname — the universal convention in python for numpy.
import numpy as np                    # fast maths across whole columns at once
import pandas as pd                   # the spreadsheet engine: load, clean, reshape
import scipy                          # statistics and scientific routines
from scipy import stats               # only the statistical-test part of scipy

# --- Charting -----------------------------------------------------------------
import matplotlib                     # the base charting engine under everything below
import matplotlib.pyplot as plt       # `plt` = the drawing commands we actually call
import seaborn as sns                 # statistical charts built on top of matplotlib
import plotly                         # interactive charts (hover, zoom, click)
import plotly.express as px           # `px` = quick one-line interactive charts
import plotly.graph_objects as go     # `go` = hand-built interactive charts, full control

# Switch off library warnings, so that anything still printed is worth reading.
warnings.filterwarnings("ignore")

# --- Where everything lives ---------------------------------------------------
# The notebook sits in a sub-folder, so we step up one level to reach the project
# root. Every path is relative to it — no hard-coded absolute paths anywhere.
# Path.cwd() = "current working directory", the folder the notebook started in.
# .name = just the last part of that path, so this asks: is that folder "notebooks"?
IS_IN_NOTEBOOKS = Path.cwd().name == "notebooks"

# The line below is a one-line if/else: `A if condition else B` means "use A when the
# condition is true, otherwise use B". .parent = the folder one level up.
PROJECT_ROOT = Path.cwd().parent if IS_IN_NOTEBOOKS else Path.cwd()

DATA_DIR = PROJECT_ROOT / "data" / "raw"          # the four source CSV files
FIG_DIR = PROJECT_ROOT / "reports" / "figures"    # charts exported for the README
# `parents=True` creates any missing folders along the way; `exist_ok=True` means no
# error if the folder is already there — which is what makes re-running this cell safe.
FIG_DIR.mkdir(parents=True, exist_ok=True)

# The four files this notebook cannot run without.
EXPECTED_INPUTS = ["shipments.csv", "products.csv", "suppliers.csv", "fx_rates.csv"]

# Stop immediately if an input is missing, rather than blaming the analysis later.
# The line below checks each expected filename and keeps only the absent ones:
#   `for f in EXPECTED_INPUTS` = take one filename at a time
#   `if not (DATA_DIR / f).exists()` = keep it when that file is NOT on disk
# The `/` joins a folder and a filename into one complete path.
missing = [f for f in EXPECTED_INPUTS if not (DATA_DIR / f).exists()]

# `assert condition, message` reads as: "if the condition is false, stop the notebook
# here and display the message". It is a guard rail, not a test.
assert not missing, (                              # if that list is not empty, stop right here
    f"Missing input file(s): {missing}\n"          # name the files that are absent
    f"Resolved PROJECT_ROOT = {PROJECT_ROOT}\n"    # say which project root we used
    f"Resolved DATA_DIR     = {DATA_DIR}\n"        # say which folder we looked in
    "Open the project folder as the VS Code workspace so the kernel starts in notebooks/."
)                                                  # ↑ and tell the reader how to fix it

# --- How tables and charts should look ----------------------------------------
pd.set_option("display.max_columns", 60)    # show all 38 shipment columns, no "..."
pd.set_option("display.width", 170)         # use the full width of the notebook
# The next line sets how decimal numbers are printed. `lambda v: ...` is a small unnamed
# rule: "take each value v and format it this way". Inside the f-string, `,.2f` means
# "thousands separators, 2 decimal places" — so 1234.5 is printed as 1,234.50.
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
sns.set_theme(style="whitegrid", context="notebook")           # consistent styling
plt.rcParams["figure.figsize"] = (11, 5)                       # default chart size, in inches
plt.rcParams["figure.dpi"] = 110                               # image sharpness

# --- How this notebook talks to you -------------------------------------------
# Every block of output opens with a titled header, so a reader always knows what they
# are looking at before they read a single number. This function prints that header.
#   `def` = define a reusable block of code, here named `section`, taking two inputs.
#   The `"""..."""` line is its description — it is what `help(section)` would show.
def section(title, explanation):
    """Print a titled header above a block of output, so the numbers explain themselves."""
    print("=" * 78)       # a piece of text repeated 78 times = a full-width horizontal rule
    print(title.upper())  # .upper() turns the title into capitals
    print("-" * 78)       # a lighter rule, directly under the title
    print(explanation)    # the plain-english "why this matters" line
    print("=" * 78)       # a matching rule, closing the block

# --- Reproducibility ----------------------------------------------------------
# A fixed "seed" makes any random step repeatable, so this notebook gives the same
# answer every time — including on someone else's machine. Nothing here uses
# randomness yet; the seed costs two lines now and prevents a surprise later.
SEED = 20260921                       # the fixed starting number, chosen once and never changed
random.seed(SEED)                     # tell python's random numbers where to start
np.random.seed(SEED)                  # tell numpy's random numbers where to start

# --- What we are running ------------------------------------------------------
# Printed now so that any strange result later can be traced to a version difference.
# `\n` inside a piece of text means "start a new line", which is how the explanation is
# wrapped below without running off the edge of the screen.
section("SETUP — LIBRARY VERSIONS",
        "What produced these results. If a number looks wrong later, check this block\n"
        "first: a different library version can change how a column is read or displayed.")

# sys.version returns something like "3.13.11 (main, ...)". `.split()` cuts that into
# words and `[0]` takes the first word — just the version number, nothing else.
print(f" python       {sys.version.split()[0]}")   # python version, trimmed to the number
print(f" pandas       {pd.__version__}")            # .__version__ = which version is installed (pandas)
print(f" numpy        {np.__version__}")            # the maths underneath pandas
print(f" scipy        {scipy.__version__}")         # the statistics library
print(f" matplotlib   {matplotlib.__version__}")    # the base charting library
print(f" seaborn      {sns.__version__}")           # the statistical-charts library
print(f" plotly       {plotly.__version__}")        # the interactive-charts library
print()                                            # print nothing = leave a blank line

section("SETUP — WHERE THE DATA WAS READ FROM",
        "The folder this notebook treats as the project root, and the folder it reads the\n"
        "four source files from. Printed in full so a mismatch is impossible to miss.")
    print(f" PROJECT_ROOT  {PROJECT_ROOT.name}")                     # folder name only, no local path
    print(f" DATA_DIR      {DATA_DIR.relative_to(PROJECT_ROOT)}")    # relative to the project root
print()                                            # blank line
print(" Ready — all four input files are present. The next cell loads them.")

## LOADING THE FILES AS THEY ARE

In [2]:
#
# In plain terms: we open the four files without letting the software change
# anything, so that what we look at next is what is really in them.
# =============================================================================

# `def` defines a reusable block of code. This one is called read_raw and takes two
# inputs: the filename, and the encoding to open it with. We write it once so all four
# files are read identically — no drift between them.
def read_raw(filename, encoding):
    """Read a CSV exactly as stored: every column as text, nothing guessed."""
    # `return` hands the result back to whoever called the function.
    return pd.read_csv(                          # hand the file to pandas, get a table back
        DATA_DIR / filename,                     # the full path to that file
        encoding=encoding,                       # the encoding the file is actually saved in
        dtype=str,                               # keep every column as text — no automatic conversion
        # pandas normally captures a list of strings ("NA", "N/A", "NULL", "-", and blank)
        # and silently replaces them with "missing". Switching that off keeps every value
        # exactly as it was written, so we can see the sentinels instead of losing them.
        keep_default_na=False,
    )


# --- Trap 1: one of the four files is not UTF-8 -------------------------------
# ! shipments.csv is Latin-1, not UTF-8. Proven here rather than assumed.
# A default read assumes UTF-8. Rather than assume, ask.
section("LOADING — TRAP 1: ONE FILE IS NOT UTF-8",
        "A default read assumes the file is saved as UTF-8. That assumption is wrong for one\n"
        "of these four files. Better to see it fail here, in one line, than to discover it\n"
        "halfway through the analysis.")
# `try:` runs the block below while watching for a failure, instead of letting it crash.
try:
    pd.read_csv(DATA_DIR / "shipments.csv")      # no encoding given → pandas assumes UTF-8
    print("shipments.csv opened as UTF-8 — unexpected, the file is Latin-1.")
# `except` catches that one specific failure and names it `e`, so we can report it
# ourselves. Any different error would still stop the notebook, as it should.
except UnicodeDecodeError as e:                  # the specific error a wrong encoding raises
    print("shipments.csv cannot be read as UTF-8:")  # what went wrong
    print(f"   {e}")                                 # the raw error: which byte, at which position
    print("   It is Latin-1 (Windows-1252) — the legacy customs-broker export.")
    print("   The accented port names are the bytes it chokes on.")   # e.g. "Lázaro Cárdenas"
print()                                          # print nothing = leave a blank line


# --- Load all four, each with the encoding it is really stored in -------------
section("LOADING — FOUR FILES, FOUR ENCODINGS",
        "Each file is opened with the encoding it was actually saved in. Nothing is converted,\n"
        "renamed or re-typed: every column arrives as text, exactly as written in the file.")

# A dictionary is a set of named entries. Each entry below is `label: (filename, encoding)`
# — the round brackets pair two values together as one item, in a fixed order.
FILES = {                                        # one entry per file: label → (filename, encoding)
    "shipments": ("shipments.csv", "latin-1"),   # legacy broker portal export
    "products":  ("products.csv",  "utf-8-sig"), # ERP export — has a BOM
    "suppliers": ("suppliers.csv", "utf-8"),     # ERP export
    "fx":        ("fx_rates.csv",  "utf-8"),     # finance export
}

# The line below reads all four files and files the results into a new dictionary. Read it
# from the inside out: `.items()` hands over each label paired with its two values;
# `for name, (fn, enc)` unpacks those three pieces; `read_raw(fn, enc)` reads that file;
# and `name: ...` stores the resulting table under the same label.
raw = {name: read_raw(fn, enc) for name, (fn, enc) in FILES.items()}

for name, df in raw.items():                     # go through the four tables one at a time
    # The braces below are layout instructions, not data. `name:10s` means "print the file
    # label left-aligned in a space 10 characters wide"; `>5,` means "print the row count
    # right-aligned in 5 characters, with thousands separators" (1458 prints as 1,458);
    # `>2` does the same for the column count in 2 characters. `shape` returns rows first,
    # then columns. The widths are chosen to fit the widest value so the four lines line up
    # and can be compared by eye.
    print(f"{name:10s} {df.shape[0]:>5,} rows x {df.shape[1]:>2} columns")
print()                                          # blank line, before the next block


# --- Trap 2: the invisible character at the start of products.csv -------------
# products.csv begins with a byte-order mark: three bytes no text editor shows you.
# ! I expected pandas to glue it onto the first column name and break products["sku"].
#   Tested: it does not — pandas strips the BOM on read (this environment: 3.0.6).
#   Other tools do not, which is why the check below shows both views.
section("LOADING — TRAP 2: THE INVISIBLE CHARACTER",
        "products.csv begins with three bytes no text editor shows you (a byte-order mark).\n"
        "I expected it to break the first column name. The three lines below are how I found\n"
        "out what really happens, instead of assuming — and the two tools disagree.")

# `.read_bytes()` reads the whole file as raw bytes, without decoding it into text.
# `[:3]` is a slice: "take the first three of them" — here, the three BOM bytes.
first_bytes = (DATA_DIR / "products.csv").read_bytes()[:3]   # the file's first three raw bytes
print("first three bytes of products.csv   :", first_bytes, "← the BOM")      # b'\xef\xbb\xbf'

# `with open(...) as f:` opens the file and closes it again automatically when the block
# ends, so a forgotten close cannot leave the file locked.
with open(DATA_DIR / "products.csv", encoding="utf-8") as f:   # open the file the plain-python way
    # Read it from the inside out: `.readline()` reads the first line; `.split(",")` cuts
    # that line into pieces at every comma; `[0]` takes the first piece — the first column
    # name. `repr()` displays it exactly as it is, invisible characters included.
    print("first header as open() sees it      :", repr(f.readline().split(",")[0]))  # line 1, first field

# `.columns` is the list of pandas' column names; `[0]` is the first of them.
print("first column as pandas sees it      :", repr(raw["products"].columns[0]))     # what pandas named it
print()                                          # blank line, before the next block

# * Worth keeping: the BOM is real, and naive tools treat it as part of the column name.
#   pandas handles it — but I only know that because I tested it instead of repeating it.


# --- Confirm nothing was converted on the way in ------------------------------
section("LOADING — PROOF THAT NOTHING WAS CONVERTED",
        "All 38 columns arrived as text, no value was turned into a 'missing value', and the\n"
        "only empty cells are ones that are empty in the file itself. These raw tables stay\n"
        "untouched for the rest of the notebook; every repair happens on a copy of them.")
s = raw["shipments"]                             # a short name for the shipments table, to save typing
print("rows x columns                          :", s.shape)               # (1458, 38) — rows first, then columns

# `.dtypes` is the data type of every column. `.astype(str)` turns those types into text;
# `set(...)` removes duplicates so each type appears once; `sorted(...)` lists them in order.
print("column types                            :", sorted(set(s.dtypes.astype(str))))  # every column is text

# `s.isna()` is a yes/no flag for every single cell, true where pandas sees a gap. The
# first `.sum()` counts those flags column by column; the second adds the columns up into
# one total. `int(...)` prints it as a whole number rather than a decimal.
print("cells pandas turned into a missing value:", int(s.isna().sum().sum()))  # target: zero

# `(s == "")` is a yes/no flag for every cell that holds empty text — the same double
# count as above. This is the honest number: blanks that are blank in the file itself.
print("cells that are genuinely empty text     :", int((s == "").sum().sum()))  # blanks, as written in the file
print()                                          # blank line, before the table

# * `raw` stays untouched for the rest of the notebook. Every repair happens in a copy,
#   so a repaired value can always be compared back to what the file actually said.
print("A first look at the shipments file as stored:")

s.head(3)                                        # `.head(3)` shows the first three rows, untouched

LOADING — TRAP 1: ONE FILE IS NOT UTF-8
------------------------------------------------------------------------------
A default read assumes the file is saved as UTF-8. That assumption is wrong for one
of these four files. Better to see it fail here, in one line, than to discover it
halfway through the analysis.
shipments.csv cannot be read as UTF-8:
   'utf-8' codec can't decode byte 0xf3 in position 873: invalid continuation byte
   It is Latin-1 (Windows-1252) — the legacy customs-broker export.
   The accented port names are the bytes it chokes on.

LOADING — FOUR FILES, FOUR ENCODINGS
------------------------------------------------------------------------------
Each file is opened with the encoding it was actually saved in. Nothing is converted,
renamed or re-typed: every column arrives as text, exactly as written in the file.
shipments  1,458 rows x 38 columns
products      45 rows x  9 columns
suppliers     15 rows x 11 columns
fx            44 rows x  4 columns

LOADING — TRA

,shipment_id,po_number,invoice_number,supplier_id,sku,order_date,etd_shipped,eta_planned,ata_arrival,customs_cleared,warehouse_receipt,carrier,vessel_voyage,incoterm,container_type,origin_port,destination_port,quantity,unit_price_usd,fob_total_usd,freight_usd,insurance_usd,net_weight_kg,gross_weight_kg,volume_cbm,hs_code,customs_value_usd,duty_igi_mxn,dta_mxn,iva_mxn,broker_fee_mxn,drayage_mxn,demurrage_mxn,storage_mxn,fx_rate_applied,qc_units_rejected,qc_inspection_notes,country_of_origin
0,IMP-2023-0001,PO-202301-001,INV-1011-202301-0001,SUP-1011,SKU-106,2023-01-02,2023-04-06,2023-05-18,2023-05-18,2023-05-20,2023-05-23,COSCO Shipping Lines,MSC PALOMA 600W,DDP,40GP,Shanghai,Manzanillo,6,"90,704.25",544225.50,9386.05,2817.3,79963.8,88721.0,107.33,8458110100,556428.85,1012962.03,81036.96,1795779.08,7063.45,60064.23,,,18.2047,0,Sin observaciones durante la inspección,China
1,IMP-2023-0002,PO-202301-002,INV-1002-202301-0001,SUP-1002,SKU-106,2023-01-02,2023-04-08,2023-05-26,2023-05-26,2023-05-31,2023-06-01,Ocean Network Express,MSC PALOMA 647W,FOB,40GP,Shekou,"Manzanillo, Colima",2,87635.07,"$175,270.14",,778.46,26654.6,30375.6,33.07,8458110100,181514.83,"MXN 330,442.30",26435.38,585808.11,8411.14,32647.23,0.0,0.0,18.2047,,,China
2,IMP-2023-0003,PO-202301-003,INV-1013-202301-0001,SUP-1013,SKU-111,2023-01-03,2023-03-17,2023-04-26,2023-04-26,2023-05-04,2023-05-05,COSCO Shipping Lines,YM WELLNESS 551E,FOB,40GP,Shanghai,Manzanillo,3,45470.74,136412.22,8144.91,674.21,5568.0,6333.5,48.29,8456110100,145231.34,264743.66,21179.49,469337.56,6418.7,34404.32,0.0,0.0,18.2291,,,China


## Profiling — what is actually in these columns

Before repairing anything, we take an inventory. For every column in all four files: how many
values are missing, how many are distinct, how short the shortest is, how long the longest is,
and one example.

Then the same question asked differently: **which cells contain something that is not what the
column claims to be** — a currency symbol, a thousands separator, a comma used as a decimal
point, an accented character, a stray space, or a placeholder like `N/A`, `-` or `#REF!`.

Nothing changes in this section. It only counts.

The point is that this table decides the order of everything that follows: whatever it flags,
we repair next, worst first.

### What to expect

- **No column in any of the four files is entirely empty.** Worth checking rather than assuming.
- Shipments is the damaged file: **17 of its 38 columns contain blanks**.
- **Nine columns carry a value that is not what the column claims to be.**

## PROFILING — AN INVENTORY OF EVERY COLUMN

In [3]:
#
# In plain terms: a stock-take. How much is missing in each column, how many
# different values it holds, and what it looks like. Nothing is changed here.
# =============================================================================

# `def` defines a reusable block of code. This one takes a table and the name of the file
# it came from, and returns one summary row per column.
def profile_columns(df, file_name):
    """One summary row per column: how much is missing, how varied, how long."""
    rows = []                                    # an empty list, to collect one summary per column
    for col in df.columns:                       # `.columns` is the list of column names — one at a time
        ser = df[col]                            # pull out that single column as its own object
        # `ser != ""` builds a yes/no flag for every row (true where the value is not empty).
        # Putting that flag inside the brackets keeps only the rows where it is true, so
        # `non_empty` is this column with the blank cells removed.
        non_empty = ser[ser != ""]
        # `.append(...)` adds one item to the list above. The item is a dictionary — a set of
        # named values — and its names become the column names of the summary table later.
        rows.append({
            "file": file_name,                   # which file this column came from
            "column": col,                       # the column's name
            # `(ser == "")` is a yes/no flag for every row; `.sum()` counts the yeses.
            # `int(...)` turns the result into a plain whole number so it prints cleanly.
            "blanks": int((ser == "").sum()),                  # how many cells are empty
            # The same count, as a share of the column: count × 100 ÷ number of rows.
            # `len(ser)` is how many rows the column has. `round(..., 1)` keeps one decimal.
            "blank_%": round(100 * (ser == "").sum() / len(ser), 1),
            # `.nunique()` counts how many different values appear. Note that an empty cell
            # counts as one of them — so a column of 1,025 blanks reports 9 distinct, not 8.
            "distinct": int(ser.nunique()),                    # how many different values
            # `.str.len()` is a text operation: the length of each value in characters.
            # `.min()` takes the shortest. The `if len(non_empty) else 0` guard means an
            # entirely empty column reports 0 instead of crashing on an empty list.
            "min_len": int(non_empty.str.len().min()) if len(non_empty) else 0,
            "max_len": int(non_empty.str.len().max()) if len(non_empty) else 0,   # the longest one
            # `.iloc[0]` picks the first row *by position* — iloc = "integer location".
            # (`.loc[0]` would instead look for the row labelled 0, which is a different job.)
            "example": non_empty.iloc[0] if len(non_empty) else "",
        })
    # Turn the list of dictionaries into a table: one dictionary per row, keys become headers.
    return pd.DataFrame(rows)


# `pd.concat([...])` stacks several tables into one. The comprehension inside calls
# profile_columns once per file and collects the four results; `ignore_index=True` then
# renumbers the rows 0, 1, 2, … instead of repeating each file's own row numbers.
profile = pd.concat([profile_columns(df, name) for name, df in raw.items()], ignore_index=True)

print(f"Inventoried {len(profile)} columns across {len(raw)} files.")   # one summary row per column, four files
print()                                          # print nothing = leave a blank line
# ! 17 of the 38 shipment columns contain blanks. That is the file we will spend our time on.
print("Columns with something missing, most affected first:")

# `profile["blanks"] > 0` is a yes/no flag for each summary row: true where that column has
# at least one blank. It keeps only those rows, then sorts them worst-first — `ascending=False`
# means largest percentage at the top.
profile[profile["blanks"] > 0].sort_values("blank_%", ascending=False)

# ? "Blank" and "zero" are not the same thing in this file. Some columns are empty because
#   nothing happened (no demurrage was charged) and others because nobody recorded anything
#   (no quality inspection was done). Deciding which is which is a judgement call, and we
#   make it deliberately in the repair blocks — not here.

Inventoried 62 columns across 4 files.

Columns with something missing, most affected first:


,file,column,blanks,blank_%,distinct,min_len,max_len,example
35,shipments,qc_units_rejected,1025,70.30,9,1,2,0
36,shipments,qc_inspection_notes,954,65.40,25,21,41,Sin observaciones durante la inspección
21,shipments,insurance_usd,602,41.30,781,3,7,2817.3
33,shipments,storage_mxn,315,21.60,76,3,8,0.0
32,shipments,demurrage_mxn,211,14.50,80,3,8,0.0
54,suppliers,longitude,2,13.30,9,5,7,114.245
53,suppliers,latitude,2,13.30,9,5,7,22.5937
43,products,unit_volume_cbm,5,11.10,38,3,5,30.55
20,shipments,freight_usd,160,11.00,1028,3,8,9386.05
10,shipments,warehouse_receipt,44,3.00,1155,5,10,2023-05-23


## PROFILING — WHAT LOOKS WRONG INSIDE THE CELLS

In [4]:

#
# In plain terms: the same stock-take, but looking for values that do not look
# like what the column is supposed to hold.
# =============================================================================

SENTINELS = ["N/A", "n/a", "NA", "-", "--", "#REF!", "#N/A", "null", "NULL", "TBD", "?"]
# ^ the placeholder values a spreadsheet or a person leaves behind when a real value is absent

s = raw["shipments"]                             # a short name for the shipments table, to save typing

# A dictionary that maps a description (the key) to a detection rule (the value). Each rule
# is a `lambda` — a small unnamed function written inline. Every one of them takes a column
# name `c` and answers with a yes/no flag for each row: true where the problem is present.
#   `lambda c: s[c] ...` reads as: "given a column name c, look at that column and ..."
# `regex=True` tells pandas the text in quotes is a *pattern* to match, not literal text to
# search for. `na=False` means "treat a missing cell as not matching" rather than erroring —
# defensive here, since we loaded the file with no missing values at all.
CHECKS = {
    # A value that changes when its outer spaces are trimmed had outer spaces to begin with.
    "leading/trailing space":  lambda c: s[c] != s[c].str.strip(),

    # Non-ASCII means "outside the first 128 characters". \x00-\x7F is that range, and the
    # ^ inside the brackets means "not". The r before the quote marks a raw string, meaning
    # "read the backslashes literally" — the normal way to write a pattern like this.
    "non-ASCII characters":    lambda c: s[c].str.contains(r"[^\x00-\x7F]", regex=True, na=False),

    # The | means "or": a dollar sign, or the letters MXN, or the letters USD. \$ is an
    # escaped dollar, because a bare $ carries special meaning inside a pattern.
    "currency symbol or code": lambda c: s[c].str.contains(r"\$|MXN|USD", regex=True, na=False),

    # \d means "any digit", so this reads: a digit, a comma, a digit — as in 1,234.
    "thousands separator":     lambda c: s[c].str.contains(r"\d,\d", regex=True, na=False),

    # The same idea, tightened: {2} means "exactly two", and $ means "at the very end of
    # the text". So this only matches a comma plus two digits at the end — as in 9.014,76.
    "comma decimal mark":      lambda c: s[c].str.contains(r"\d,\d{2}$", regex=True, na=False),

    # `.isin(...)` asks a different question: is this value one of the strings in that list?
    "known sentinel value":    lambda c: s[c].isin(SENTINELS),
}

# The statement below builds the summary table, and it is dense. Read it inside out:
#   inner  [int(check(c).sum()) for c in s.columns]
#          for each column name c, run that check on the column (giving a yes/no flag per
#          row), count the yeses, and turn the count into a plain number.
#   outer  {label: [...] for label, check in CHECKS.items()}
#          turn those lists into a dictionary of label → list of counts.
#   then   index=s.columns labels the rows with the data's column names, so the finished
#          table is transposed: one row per data column, one column per check.
suspicious = pd.DataFrame(
    {label: [int(check(c).sum()) for c in s.columns] for label, check in CHECKS.items()},
    index=s.columns,
)
# `axis=1` means "add across each row", left to right. (`axis=0` would add down each column
# instead.) So this column holds, for each data column, how many problem flags it collected.
suspicious["total"] = suspicious.sum(axis=1)

# * This is the roadmap. Every line below becomes a repair step later in the notebook.
for label in CHECKS:                             # a dictionary loops over its keys — the check descriptions
    cells = int(suspicious[label].sum())         # total flags in that one check's column
    cols = int((suspicious[label] > 0).sum())    # how many data columns collected at least one flag
    # `label:26s` pads the description; `>5,` and `>2` right-align the two counts, with commas.
    print(f"{label:26s} {cells:>5,} cells across {cols:>2} column(s)")
print()                                          # print nothing = leave a blank line

# ! 1,033 cells carry a currency symbol or code, and 1,291 carry a thousands separator.
#   Every one of those is text pretending to be a number — pandas will not do arithmetic on them.
# ? 500 cells contain non-ASCII characters: 271 in destination_port, 229 in the quality notes.
#   In a port name that is a joining problem. In a Spanish note it is simply Spanish.
#   Same flag, two completely different decisions.

# ! Nine of 38 columns carry something that is not what they claim to be.
print(f"{int((suspicious['total'] > 0).sum())} of {len(suspicious)} columns flagged:")

# `suspicious["total"] > 0` flags each row that collected any flag at all; keeping those and
# sorting by the total, worst first, puts the columns needing the most repair at the top.
suspicious[suspicious["total"] > 0].sort_values("total", ascending=False)

leading/trailing space        27 cells across  2 column(s)
non-ASCII characters         500 cells across  2 column(s)
currency symbol or code    1,033 cells across  3 column(s)
thousands separator        1,291 cells across  4 column(s)
comma decimal mark            56 cells across  2 column(s)
known sentinel value           6 cells across  2 column(s)

9 of 38 columns flagged:


,leading/trailing space,non-ASCII characters,currency symbol or code,thousands separator,comma decimal mark,known sentinel value,total
fob_total_usd,0,0,731,731,0,0,1462
unit_price_usd,0,0,157,412,53,0,622
duty_igi_mxn,0,0,145,145,0,0,290
destination_port,0,271,0,0,0,0,271
qc_inspection_notes,0,229,0,0,0,0,229
incoterm,17,0,0,0,0,4,21
supplier_id,10,0,0,0,0,0,10
broker_fee_mxn,0,0,0,3,3,0,6
dta_mxn,0,0,0,0,0,2,2


## DATES — WHAT FORMAT IS EACH VALUE ACTUALLY IN?

In [5]:
#
# In plain terms: the same date can be written several ways, and a computer
# cannot tell 03/04/2024 (4 March) from 03/04/2024 (3 April) on its own. Before
# we can put shipments in order, we need to know which format each value uses.
# =============================================================================

# The six date columns, in the order events happen. The brackets below stay open across
# two lines, which is what lets python accept a list split like this without a backslash.
DATE_COLUMNS = ["order_date", "etd_shipped", "eta_planned",
                "ata_arrival", "customs_cleared", "warehouse_receipt"]

# A dictionary mapping a human description to the pattern that recognises that format.
# Inside these patterns: ^ means "starts here", $ means "ends here", \d means "any digit",
# {4} means "exactly four of the previous thing", and {1,2} means "one or two of them".
PATTERNS = {
    "ISO  YYYY-MM-DD":                r"^\d{4}-\d{2}-\d{2}$",   # 2024-03-04
    "slash DD/MM/YYYY or MM/DD/YYYY": r"^\d{1,2}/\d{1,2}/\d{4}$",  # 04/03/2024 — ambiguous
    "dash  DD-MM-YYYY":               r"^\d{1,2}-\d{1,2}-\d{4}$",  # 04-03-2024
    "Excel serial number":            r"^\d{5}$",               # 45355 — days since 1899
}

rows = []                                        # an empty list, to collect one row per date column
for col in DATE_COLUMNS:                         # work through the six columns, one at a time
    ser = s[col]                                 # that column on its own
    non_empty = ser[ser != ""]                   # the same column with blank cells removed
    # Start this column's row as a dictionary. The keys will become the table's headers.
    counts = {"column": col, "empty": int((ser == "").sum())}
    for label, pattern in PATTERNS.items():      # test the column against each of the four formats
        # `.str.match` asks "does this value match the pattern from the start?" — unlike
        # `.str.contains`, which would accept a match anywhere in the text. It answers with a
        # yes/no flag per row, `.sum()` counts the yeses, and `int(...)` makes it a plain number.
        # No `na=False` is needed here: blanks were filtered out of `non_empty` a moment ago.
        counts[label] = int(non_empty.str.match(pattern).sum())
    # Anything that matched none of the four. `len(non_empty)` is the total in the column;
    # the `sum(...)` adds up the four counts by looping over the pattern names as keys.
    # If two patterns had matched the same value this would go negative — it comes out 0,
    # which also proves the four formats do not overlap.
    counts["unrecognised"] = len(non_empty) - sum(counts[l] for l in PATTERNS)
    rows.append(counts)                          # add this column's row to the list

# Turn the list of dictionaries into a table. `.set_index("column")` promotes that field
# from a data column to the row labels, so each row is named after the date column it describes.
date_formats = pd.DataFrame(rows).set_index("column")

# * Four different formats in six columns — because this file is a merge of two systems.
#   The ERP export and the broker portal each wrote dates their own way.
# ! The slash format is the dangerous one: `03/04/2024` is a valid date in BOTH day-first
#   and month-first reading, so the pattern alone cannot tell us which it is.
date_formats                                     # the table renders below this line

,empty,ISO YYYY-MM-DD,slash DD/MM/YYYY or MM/DD/YYYY,dash DD-MM-YYYY,Excel serial number,unrecognised
column,,,,,,
order_date,0,582,735,60,81,0
etd_shipped,0,582,735,60,81,0
eta_planned,28,571,720,60,79,0
ata_arrival,12,578,728,59,81,0
customs_cleared,36,568,717,58,79,0
warehouse_receipt,44,568,711,58,77,0


## DATES — RESOLVING THE AMBIGUOUS SLASH DATES FROM INTERNAL EVIDENCE

In [6]:
#
# In plain terms: 03/04/2024 could be 4 March or 3 April. But the file tells us
# which, if we look. Each row came from one system, and that system wrote every
# date in that row the same way. So if one date in a row is unambiguous — say
# 13/04/2023, which can only be day-first — then every other slash date in that
# row follows the same convention. Evidence, not guesswork.
# =============================================================================

# The pattern for a slash date, reused by the checks below. Written once here so every
# test in this cell is asking exactly the same question.
SLASH = r"^\d{1,2}/\d{1,2}/\d{4}$"

# A function that takes one row and returns a single word describing how its slash dates
# should be read. It returns a *label*, not a yes/no answer, because there are four
# possible outcomes and each one leads to a different decision later.
def slash_verdict(row):
    """Decide from one row's slash values whether it is day-first or month-first."""
    # Keep only the values in this row that look like slash dates. Three things happen here:
    #   `isinstance(v, str)` asks "is this value text?" — defensive, since every value in
    #      this file is text, but it stops the function breaking on a different file later.
    #   `re.match(SLASH, v)` asks "does it match the pattern from the start?" — and returns
    #      a match object (which counts as true) or None (which counts as false).
    #   The whole line is a list comprehension: build a list by keeping the values that pass.
    slash_values = [v for v in row if isinstance(v, str) and re.match(SLASH, v)]

    # A guard clause. An empty list counts as false, so `not slash_values` reads "if the
    # list is empty". Returning early avoids running the rest on a row with nothing to judge.
    if not slash_values:
        return "not applicable (no slash dates)"

    day_first = month_first = False              # two flags, both starting as "no evidence yet"
    for value in slash_values:                   # look at each slash date in this row
        # This line unpacks three steps into two names. Read it inside out:
        #   `value.split("/")` cuts "13/04/2023" into a list: ["13", "04", "2023"]
        #   `[:2]` slices off the first two parts, dropping the year
        #   `int(p)` turns each remaining piece of text into a number
        #   `first, second = ...` assigns the two results to two names at once
        first, second = (int(p) for p in value.split("/")[:2])

        # There are only twelve months, so a number above 12 cannot be a month — it must be
        # a day. One such value is enough to settle this row, which is the whole method.
        if first > 12:      # 27/03/2024 — 27 cannot be a month → day first
            day_first = True
        if second > 12:     # 03/27/2024 — 27 cannot be a month → month first
            month_first = True

    # Evidence pointing both ways would mean two different systems wrote one row, which
    # contradicts the assumption the whole method rests on. We check for it rather than
    # assume it, and the count of zero later is what validates the method.
    if day_first and month_first:
        return "contradictory"          # impossible if one system wrote the row
    if day_first:
        return "day-first"
    if month_first:
        return "month-first"
    return "unresolved"                 # every slash value in the row reads either way


# `.apply(slash_verdict, axis=1)` runs that function once per row. `axis=1` means "across
# each row"; `axis=0` would instead go down each column. The result is one verdict per row,
# held in the same order as the rows themselves.
row_verdict = s[DATE_COLUMNS].apply(slash_verdict, axis=1)

# * About half the rows carry no slash dates at all. Those need no resolution —
#   ISO, dash and Excel serial values each have only one possible reading.
# ! Zero contradictory rows. That is the check that validates the whole method:
#   if the "one row, one system" assumption were false, we would see rows whose
#   dates point both ways at once.
# ? 17 rows stay unresolved: every date in them is ambiguous. They need a decision,
#   and the next cell settles them by testing which reading produces a plausible
#   order of events — ordered before shipped, shipped before arrived, and so on.

# Pick one month-first row to show the reasoning in the open. `.index` gives the row labels
# that survived the filter, and `[0]` takes the first of them.
example = row_verdict[row_verdict == "month-first"].index[0]
print(f"Worked example — row {example}:")
# `.loc[row, columns]` selects by *label*: this row, these columns. `.to_string()` prints it
# as plain text rather than as a formatted table, so it sits neatly inside the output.
print(s.loc[example, DATE_COLUMNS].to_string())
print("\nOne value in that row is decisive:")    # \n starts a new line before this sentence
for col in DATE_COLUMNS:                         # walk the six date columns of that row
    value = s.loc[example, col]
    if re.match(SLASH, value):                   # only the ones written as slash dates
        first, second = (int(p) for p in value.split("/")[:2])
        # A chained conditional, read left to right: the first condition that is true wins.
        # `A if cond1 else B if cond2 else C` means "use A if cond1, otherwise B if cond2,
        # otherwise C". The padded widths keep the three possible labels in a neat column.
        verdict_note = ("DECIDES month-first" if second > 12
                        else "DECIDES day-first" if first > 12 else "ambiguous")
        print(f"   {col:20s} {value:12s} -> {verdict_note}")
print()                                          # blank line, before the summary

# `.value_counts()` counts how many times each distinct verdict appears, most frequent first.
# `.to_frame("rows")` turns that into a one-column table and names the column "rows".
verdict_summary = row_verdict.value_counts().to_frame("rows")
# Assigning to a column name that does not exist yet creates it: count × 100 ÷ number of
# rows, rounded to one decimal place — the share of all shipments each verdict accounts for.
verdict_summary["share_%"] = (100 * verdict_summary["rows"] / len(s)).round(1)
verdict_summary                                 # the summary table renders below

Worked example — row 10:
order_date           01/08/2023
etd_shipped          04/13/2023
eta_planned          05/11/2023
ata_arrival          05/11/2023
customs_cleared      05/13/2023
warehouse_receipt    05/16/2023

One value in that row is decisive:
   order_date           01/08/2023   -> ambiguous
   etd_shipped          04/13/2023   -> DECIDES month-first
   eta_planned          05/11/2023   -> ambiguous
   ata_arrival          05/11/2023   -> ambiguous
   customs_cleared      05/13/2023   -> DECIDES month-first
   warehouse_receipt    05/16/2023   -> DECIDES month-first



,rows,share_%
not applicable (no slash dates),723,49.60
day-first,414,28.40
month-first,304,20.90
unresolved,17,1.20


## DATES — SETTLING THE LAST ROWS: DOES THE ORDER OF EVENTS MAKE SENSE?

In [7]:
#
# In plain terms: 17 rows have every date written ambiguously, so nothing inside
# them decides the format. But events have an order that shipping enforces — a
# machine cannot clear customs before it has arrived, or arrive before it was
# shipped. So we read each row both ways and keep the reading that produces a
# sequence that could actually have happened.
# =============================================================================

# The three unambiguous formats, named once each so the parsing below can refer to them.
ISO = r"^\d{4}-\d{2}-\d{2}$"                     # 2024-03-04
DASH = r"^\d{1,2}-\d{1,2}-\d{4}$"                # 04-03-2024, always day first
SERIAL = r"^\d{5}$"                              # 45355, a day number counted by Excel

# A list of pairs, each read as "the first must happen before the second". The round brackets
# make each pair a single item — a tuple — so the list holds five pairs, not ten loose names.
REQUIRED_ORDER = [
    ("order_date",      "etd_shipped"),       # it cannot leave before it was ordered
    ("etd_shipped",     "eta_planned"),       # it cannot be due before it left
    ("etd_shipped",     "ata_arrival"),       # it cannot arrive before it left
    ("ata_arrival",     "customs_cleared"),   # it cannot clear before it arrived
    ("customs_cleared", "warehouse_receipt"), # it cannot be received before it cleared
]
# * Note what is deliberately NOT in that list: (eta_planned, ata_arrival).
#   Arriving EARLIER than planned is normal in shipping, so there is no ordering
#   rule between the plan and the reality. Requiring one would mark every early
#   arrival as corrupt — a rule that looks sensible and is wrong. I wrote it that
#   way first and it condemned a perfectly good row.

# Reads one row's dates under one convention. Returns a dictionary of real dates, or None
# when that reading cannot be true. `convention` is the word decided in the cell above —
# "day-first" or "month-first" — which is what makes the ambiguous values readable.
def parse_row_dates(values, convention):
    """Read one row's date values under one convention. None if unreadable."""
    parsed = {}                                  # will hold column name → real date
    for col, value in values.items():            # on a row, `.items()` gives name + value pairs
        if value == "":
            continue                             # `continue` jumps to the next value, skipping this one
        try:
            if re.match(ISO, value):             # already in the format we want
                parsed[col] = pd.Timestamp(value)      # turn the text into a real date object
            elif re.match(SERIAL, value):        # an Excel day number
                # Excel counts days from 30 December 1899, so add that many days to that
                # starting point. `pd.Timedelta(days=…)` is a length of time; adding a length
                # of time to a date produces another date.
                parsed[col] = pd.Timestamp("1899-12-30") + pd.Timedelta(days=int(value))
            elif re.match(DASH, value):          # 04-03-2024 — written day first, no ambiguity
                day, month, year = value.split("-")            # three pieces, in that order
                parsed[col] = pd.Timestamp(f"{year}-{month}-{day}")   # rebuilt as ISO
            elif re.match(SLASH, value):         # the ambiguous one
                first, second, year = value.split("/")
                # The one-line if/else again: assign two names at once, in whichever order
                # matches the convention being tested.
                day, month = (first, second) if convention == "day-first" else (second, first)
                parsed[col] = pd.Timestamp(f"{year}-{month}-{day}")
            else:
                return None                      # an unfamiliar format — this reading fails
        except Exception:
            # Catching every error is usually a smell. Here it is the point: if any value
            # cannot become a real date (a month of 13, for instance), then this reading is
            # impossible — and "impossible" is exactly the answer we want. The cost is that a
            # genuine bug in this block would land here too, which is why the result is
            # checked against real rows rather than trusted.
            return None
    return parsed                                # hand back everything that was readable

def chain_is_ordered(parsed):
    """True / False / None. None means the row has too few dates to judge."""
    # Keep only the pairs where BOTH dates exist in this row. `a in parsed` asks whether
    # that column name is one of the keys we collected.
    pairs = [(a, b) for a, b in REQUIRED_ORDER if a in parsed and b in parsed]
    if not pairs:
        return None                              # nothing to judge — an honest "unknown"
    # `all(...)` is True only if every single comparison is True. Comparing two dates with
    # `<=` is a plain chronological comparison: the earlier date is the smaller one.
    return all(parsed[a] <= parsed[b] for a, b in pairs)


# The row labels of the 17 rows that carry no internal evidence — the leftovers from the
# cell above. `.index` hands us those labels.
UNRESOLVED = row_verdict[row_verdict == "unresolved"].index

resolutions = []                                 # one summary dictionary per row, collected here
for idx in UNRESOLVED:                           # work through those rows one at a time
    row = s.loc[idx, DATE_COLUMNS]               # that row, restricted to the six date columns
    day_first = parse_row_dates(row, "day-first")      # try reading it one way
    month_first = parse_row_dates(row, "month-first")  # and try the other way

    # Read these two lines as: "the reading must have succeeded, AND the order of events it
    # produces must be possible". `and` stops at the first part that fails, so None is never
    # handed to the test. `is True` is stricter than merely being true: the test can answer
    # True, False or None, and only True is good enough here.
    ok_day = day_first is not None and chain_is_ordered(day_first) is True
    ok_month = month_first is not None and chain_is_ordered(month_first) is True

    # Four possible outcomes. `elif` means "otherwise, if" — only the first match runs.
    if ok_day and not ok_month:
        decision = "day-first"                   # only day-first gives a possible sequence
    elif ok_month and not ok_day:
        decision = "month-first"                 # only month-first does
    elif ok_day and ok_month:
        decision = "undecided — both readings possible"
    else:
        decision = "undecided — neither reading possible"

    # One summary row per shipment: which row, what we decided, and both test results — kept
    # so a reader can see the evidence rather than only the verdict.
    resolutions.append({"row": idx, "decision": decision,
                        "day_first_fits": ok_day, "month_first_fits": ok_month})

# Turn the list of dictionaries into a table, using the row label as the table's index.
resolutions = pd.DataFrame(resolutions).set_index("row")

# ! 13 of the 17 are now settled — by evidence from outside the ambiguous values.
# ? 4 rows survive even this test: both readings produce a sequence that could have
#   happened. They need a different kind of evidence, and the next cell finds some
#   already sitting in the PO number.

# `.isin([...])` asks "is this value one of these two words?" and counts the yeses.
settled = resolutions["decision"].isin(["day-first", "month-first"]).sum()
print(f"Settled by the order of events: {settled} of {len(resolutions)}")
print()                                          # print nothing = leave a blank line

resolutions                                      # the table renders below this line

Settled by the order of events: 13 of 17



,decision,day_first_fits,month_first_fits
row,,,
151,day-first,True,False
160,day-first,True,False
233,undecided — both readings possible,True,True
316,day-first,True,False
395,undecided — both readings possible,True,True
433,undecided — both readings possible,True,True
451,day-first,True,False
632,month-first,False,True
635,day-first,True,False
